# Regularization, cell by cell

Goodfellow et al., [ch. 7](https://www.deeplearningbook.org/contents/regularization.html). Notes: `NOTES.md`.

## How to click through this

1. Open **this file** in Cursor (or VS Code / Jupyter). You should see cells, not raw JSON.
2. Top-right: pick the **Python 3** kernel that has `numpy / sklearn / torch` (`pip install -r regularization/requirements.txt`).
3. Click the first code cell (setup). **Shift+Enter** = run it and jump to the next cell. That's the loop.
4. **Ctrl/Cmd+Enter** = re-run the *current* cell in place. Use this after you change a knob (`ALPHA`, `WD`, …).
5. Geometry + linear (through §6) are instant and already show **L1 punching weights to 0**. Fashion-MNIST cells train a net — a few seconds each.

Do **not** need to Run All. Skip any MLP cell you're bored of. Re-run setup after a kernel restart.

Default `EPOCHS=8`, `N_TRAIN=800`. Bump them if you want `run.py`-sized gaps.

## 0. Setup

Run once. Works from repo root or from `regularization/`.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = "retina"

import os, sys
from pathlib import Path

HERE = Path.cwd().resolve()
if (HERE / "regularization" / "train.py").exists():
    HERE = HERE / "regularization"
elif not (HERE / "train.py").exists():
    raise FileNotFoundError(f"can't find train.py from {Path.cwd()}")
os.chdir(HERE)
sys.path.insert(0, str(HERE))
print("cwd:", HERE)
print("Shift+Enter runs a cell. Ctrl+Enter re-runs it after you tweak a knob.")

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.linear_model import Lasso, LinearRegression, LogisticRegression, Ridge

from closed_form import l1_soft_threshold, l2_shrink_diag, l2_shrink_eigen, ridge_normal_equation
from data import (
    FASHION_LABELS,
    load_cancer_split,
    load_diabetes_split,
    load_fashion_split,
    shift_images,
)
from models import MLP, TinyCNN, fgsm, n_params, weight_l1
from train import (
    TrainConfig,
    bagged_proba,
    bootstrap_indices,
    metrics_for,
    train_classifier,
)
import viz

plt.rcParams.update({"figure.figsize": (6.2, 4.0), "figure.dpi": 110})

EPOCHS = 8
N_TRAIN = 800
N_VAL = 800
N_TEST = 1500
LR = 3e-3
HIDDEN = (256, 256)
SEED = 0

BOARD: dict[str, dict] = {}
MODELS: dict[str, torch.nn.Module] = {}
HISTS: dict = {}


def show_board() -> None:
    viz.scoreboard(BOARD)


def param_l2(model: torch.nn.Module) -> float:
    tot = 0.0
    for p in model.parameters():
        if p.ndim > 1:
            tot += float(p.detach().pow(2).sum())
    return tot ** 0.5


def plot_hist(hist, title: str = "") -> None:
    fig, ax = plt.subplots(1, 2, figsize=(9.2, 3.3))
    ax[0].plot(hist.train_acc, label="train")
    ax[0].plot(hist.val_acc, label="val")
    ax[0].set_title("accuracy")
    ax[0].set_xlabel("epoch")
    ax[0].legend()
    ax[1].plot(hist.train_loss, label="train")
    ax[1].plot(hist.val_loss, label="val")
    if getattr(hist, "best_epoch", None) is not None:
        ax[1].axvline(hist.best_epoch, ls=":", color="k", lw=1)
    ax[1].set_title("loss")
    ax[1].set_xlabel("epoch")
    ax[1].legend()
    fig.suptitle(title, y=1.02)
    plt.show()


def fit_mlp(name: str, dropout: float = 0.0, **cfg) -> tuple:
    data = DATA
    torch.manual_seed(cfg.get("seed", SEED))
    model = MLP(data.n_features, data.n_classes, hidden=HIDDEN, dropout=dropout)
    tc = TrainConfig(
        epochs=cfg.get("epochs", EPOCHS),
        lr=cfg.get("lr", LR),
        batch_size=64,
        optimizer="adamw",
        weight_decay=cfg.get("weight_decay", 0.0),
        l1=cfg.get("l1", 0.0),
        activation_l1=cfg.get("activation_l1", 0.0),
        input_noise=cfg.get("input_noise", 0.0),
        label_smoothing=cfg.get("label_smoothing", 0.0),
        early_stop_patience=cfg.get("early_stop_patience"),
        early_stop_min_epoch=cfg.get("early_stop_min_epoch", 4),
        restore_best=cfg.get("restore_best", False),
        adversarial_eps=cfg.get("adversarial_eps", 0.0),
        seed=cfg.get("seed", SEED),
    )
    hist = train_classifier(model, data.X_train, data.y_train, data.X_val, data.y_val, tc)
    met = metrics_for(
        model, data.X_train, data.y_train, data.X_val, data.y_val, data.X_test, data.y_test
    )
    BOARD[name] = {
        "train": met.train_acc,
        "val": met.val_acc,
        "test": met.test_acc,
        "gap": met.gap,
    }
    MODELS[name] = model
    HISTS[name] = hist
    print(
        f"{name:18} train={met.train_acc:.3f}  val={met.val_acc:.3f}  "
        f"test={met.test_acc:.3f}  gap={met.gap:.3f}  stopped@{hist.stopped_epoch}"
        f"  ||W||_2={param_l2(model):.1f}"
    )
    plot_hist(hist, name)
    return model, hist, met


## 1. What we're doing

Regularization = any change intended to **cut generalization error, not training error**.

\[
\tilde{J}(\theta; X, y) = J(\theta; X, y) + \alpha\,\Omega(\theta)
\]

Penalize **weights, not biases** (ch. 7.1). A bias is one number per unit.

## 2. \(L_2\) geometry — Fig 7.1

Quadratic \(J\) around \(w^*\), Hessian \(H=\mathrm{diag}(\lambda_1,\lambda_2)\) with \(\lambda_1\ll\lambda_2\):

\[
\tilde{w} = Q\,(\Lambda+\alpha I)^{-1}\Lambda\,Q^\top w^*, \qquad
\tilde{w}_i = \frac{\lambda_i}{\lambda_i+\alpha}\,w_i^*
\]

**Look at the figure:** the blue ellipses are level sets of \(J\). Red dashed = \(L_2\) balls. The square (\(\tilde{w}\)) is dragged toward 0 **along the poorly determined axis** \(w_1\).

**Knob:** `ALPHA`. Re-run (Ctrl+Enter). Watch \(\tilde{w}\) slide.

In [ ]:
ALPHA = 0.55  # try 0.05, 0.55, 2.0

w_star = np.array([1.8, 0.7])
H = np.diag([0.15, 2.4])  # λ1 small → w1 poorly determined
w_t = l2_shrink_eigen(w_star, H, ALPHA)
scale = np.diag(H) / (np.diag(H) + ALPHA)
print("w*        ", w_star)
print("w_tilde   ", w_t)
print("λ/(λ+α)   ", scale)

w1 = np.linspace(-0.6, 2.4, 240)
w2 = np.linspace(-1.2, 1.8, 240)
W1, W2 = np.meshgrid(w1, w2)
delta = np.stack([W1 - w_star[0], W2 - w_star[1]], axis=-1)
J = 0.5 * np.einsum("...i,ij,...j->...", delta, H, delta)
R = 0.5 * (W1**2 + W2**2)

fig, ax = plt.subplots(figsize=(5.8, 5.2), layout="constrained")
ax.contour(W1, W2, J, levels=8, colors="#1f77b4")
ax.contour(W1, W2, R, levels=8, colors="#d62728", linestyles="--")
ax.plot(*w_star, "o", color="#1f77b4", ms=9, label=r"$w^*$ unregularized")
ax.plot(*w_t, "s", color="#d62728", ms=9, label=r"$\tilde{w}$ with $L_2$")
ax.annotate(
    "", xy=w_t, xytext=w_star,
    arrowprops=dict(arrowstyle="->", color="#333", lw=1.4),
)
ax.axhline(0, color="#ccc", lw=0.6)
ax.axvline(0, color="#ccc", lw=0.6)
ax.set_xlabel(r"$w_1$ poorly determined ($\lambda=0.15$)")
ax.set_ylabel(r"$w_2$ well determined ($\lambda=2.4$)")
ax.set_aspect("equal")
ax.legend(frameon=False, loc="lower right")
ax.set_title(rf"$L_2$ geometry, $\alpha={ALPHA}$")
plt.show()


SGD view (eq 7.5): \(w \leftarrow (1-\epsilon\alpha)w - \epsilon\nabla J\). Multiplicative shrink, then the data step. MAP: Gaussian prior.

## 3. \(L_1\) vs \(L_2\) in 1-D — the dead zone

\[
\tilde{w}_i^{\mathrm{L1}} = \mathrm{sign}(w_i^*)\max\bigl(|w_i^*| - \alpha/H_{ii},\, 0\bigr)
\qquad
\tilde{w}_i^{\mathrm{L2}} = \frac{H_{ii}}{H_{ii}+\alpha}\,w_i^*
\]

**Look at the orange band:** inside \(|w^*| < \alpha/H\), L1 output is **exactly 0**. L2 never hits the axis (unless \(w^*=0\)).

**Knob:** `ALPHA` — the dead zone widens.

In [ ]:
ALPHA = 1.0  # try 0.2, 1.0, 2.5

w_star = np.linspace(-3, 3, 400)
H = np.ones_like(w_star)
thresh = ALPHA / H[0]
l1 = l1_soft_threshold(w_star, H, ALPHA)
l2 = l2_shrink_diag(w_star, H, ALPHA)

fig, ax = plt.subplots(figsize=(6.8, 4.4), layout="constrained")
ax.axvspan(-thresh, thresh, color="#ff7f0e", alpha=0.18, label=rf"L1 dead zone $|w^*|<\alpha/H={thresh:.2f}$")
ax.plot(w_star, w_star, color="#bbb", label=r"$w^*$")
ax.plot(w_star, l2, color="#1f77b4", lw=2, label=r"$L_2$  $\lambda/(\lambda+\alpha)$")
ax.plot(w_star, l1, color="#ff7f0e", lw=2, label=r"$L_1$  soft-threshold")
ax.scatter([-thresh, thresh], [0, 0], color="#d62728", zorder=5, s=40)
ax.axhline(0, color="#aaa", lw=0.6)
ax.axvline(0, color="#aaa", lw=0.6)
ax.set_xlabel(r"$w^*$")
ax.set_ylabel(r"$\tilde{w}$")
ax.set_title(rf"shrinkage, $\alpha={ALPHA}$  — L1 is identically 0 in the band")
ax.legend(frameon=False, fontsize=8)
plt.show()

print("L2 at w*=0.3 →", float(l2_shrink_diag(np.array([0.3]), np.array([1.0]), ALPHA)[0]))
print("L1 at w*=0.3 →", float(l1_soft_threshold(np.array([0.3]), np.array([1.0]), ALPHA)[0]), "  (zero if 0.3 < α)")


## 4. Penalties as constraints (ch. 7.2)

\(\tilde{J}=J+\alpha\Omega\) is the Lagrangian for \(\min J\) s.t. \(\Omega(\theta)\le k\).

**Look at the red dots:** \(L_2\) optimum sits *off-axis* on the disk. \(L_1\) optimum sits on a **vertex** of the diamond → that coordinate is exactly 0.

In [ ]:
theta = np.linspace(0, 2 * np.pi, 400)
fig, axes = plt.subplots(1, 2, figsize=(8.6, 4.0), layout="constrained")
axes[0].plot(np.cos(theta), np.sin(theta), color="#1f77b4", lw=2)
axes[0].plot([0.72], [0.69], "o", color="#d62728", ms=9)
axes[0].annotate("off-axis\n(both coords live)", xy=(0.72, 0.69), xytext=(-0.2, 1.05),
                 fontsize=8, arrowprops=dict(arrowstyle="->", color="#444"))
axes[0].set_title(r"$L_2$ ball $\|w\|_2 \leq k$")
diamond = np.array([[1, 0], [0, 1], [-1, 0], [0, -1], [1, 0]], float)
axes[1].plot(diamond[:, 0], diamond[:, 1], color="#ff7f0e", lw=2)
axes[1].plot([0.0], [1.0], "x", color="#d62728", ms=12, mew=2)
axes[1].annotate("$w_1 = 0$", xy=(0.0, 1.0), xytext=(0.45, 0.55),
                 fontsize=9, color="#d62728", arrowprops=dict(arrowstyle="->", color="#d62728"))
axes[1].set_title(r"$L_1$ diamond $\|w\|_1 \leq k$")
w1 = np.linspace(-1.4, 1.4, 200)
W1, W2 = np.meshgrid(w1, w1)
J = 0.5 * ((W1 - 1.15) ** 2 / 0.7 + (W2 - 1.05) ** 2 / 0.55)
for ax in axes:
    ax.contour(W1, W2, J, levels=6, colors="#888", linewidths=0.8)
    ax.axhline(0, color="#ccc", lw=0.6)
    ax.axvline(0, color="#ccc", lw=0.6)
    ax.set_aspect("equal")
    ax.set_xlim(-1.4, 1.4)
    ax.set_ylim(-1.4, 1.4)
    ax.set_xlabel(r"$w_1$")
    ax.set_ylabel(r"$w_2$")
plt.show()


## 5. Linear: diabetes — Lasso actually zeros coefficients

10 real features. **Red × = exact zero.** OLS/Ridge stay dense; Lasso drops `s1`/`s4`-class junk and keeps `bmi` + `s5`.

Then the path plot: crank \(\alpha\) and watch lines collapse onto the axis.

**Knobs:** `RIDGE_ALPHA`, `LASSO_ALPHA`.

In [ ]:
RIDGE_ALPHA = 2.0
LASSO_ALPHA = 0.8

d = load_diabetes_split()
ols = LinearRegression().fit(d.X_train, d.y_train)
ridge = Ridge(alpha=RIDGE_ALPHA).fit(d.X_train, d.y_train)
lasso = Lasso(alpha=LASSO_ALPHA, max_iter=20000).fit(d.X_train, d.y_train)

def mse(m):
    return float(np.mean((m.predict(d.X_test) - d.y_test) ** 2))

for name, m in [("OLS", ols), ("Ridge", ridge), ("Lasso", lasso)]:
    z = int(np.sum(np.abs(m.coef_) < 1e-6))
    dead = [d.feature_names[i] for i, v in enumerate(m.coef_) if abs(v) < 1e-6]
    print(f"{name:6}  test MSE={mse(m):8.1f}  zeros={z}/{m.coef_.size}  dropped={dead}")

viz.stems_with_zeros(
    {"OLS": ols.coef_, f"Ridge α={RIDGE_ALPHA}": ridge.coef_, f"Lasso α={LASSO_ALPHA}": lasso.coef_},
    d.feature_names,
    title="diabetes weights — red × are exact zeros (L1)",
)


Same data, sweep \(\alpha\). Left: each coefficient's path. Right: zero-count vs test MSE. This is the picture people mean by "L1 does feature selection".

In [ ]:
viz.lasso_zero_path(d.X_train, d.y_train, d.X_test, d.y_test, d.feature_names)


Closed form check: unregularized least squares with a bias column should match sklearn OLS (cosine ~ 1).

In [ ]:
Xb = np.c_[d.X_train, np.ones(len(d.X_train))]
closed = ridge_normal_equation(Xb, d.y_train, alpha=0.0)
align = float(
    np.dot(closed[:-1], ols.coef_)
    / (np.linalg.norm(closed[:-1]) * np.linalg.norm(ols.coef_) + 1e-12)
)
print("OLS vs closed-form cosine:", round(align, 6))


## 6. Logistic \(L_1\) on breast_cancer

30 features, tiny train split. Unregularized interpolates. L1 keeps a handful (usually `worst concave points`).

sklearn 1.8+: `l1_ratio=0` is \(L_2\), `l1_ratio=1` is \(L_1\), `C=np.inf` is none. Smaller C = more penalty.

**Knob:** `C_L1` — then the path plot inverts C so "left = more L1".

In [ ]:
C_L1 = 0.8   # try 0.2 (very sparse) vs 5.0 (almost dense)
C_L2 = 0.5

c = load_cancer_split()
models = {
    "none": LogisticRegression(C=np.inf, max_iter=4000, random_state=0),
    "L2": LogisticRegression(C=C_L2, l1_ratio=0.0, max_iter=4000, random_state=0),
    "L1": LogisticRegression(C=C_L1, l1_ratio=1.0, solver="saga", max_iter=8000, random_state=0),
}
for name, m in models.items():
    m.fit(c.X_train, c.y_train)
    z = int(np.sum(np.abs(m.coef_) < 1e-4))
    print(
        f"{name:5}  train={m.score(c.X_train, c.y_train):.3f}  "
        f"test={m.score(c.X_test, c.y_test):.3f}  zeros={z}/{m.coef_.size}"
    )

w = models["L1"].coef_.ravel()
print("\nL1 survivors:")
for i in np.argsort(-np.abs(w)):
    if abs(w[i]) < 1e-4:
        continue
    print(f"  {c.feature_names[i]:24}  {w[i]:+.3f}")

viz.stems_with_zeros(
    {"logreg L1": w},
    c.feature_names,
    title="breast_cancer L1 — red × dropped features",
)


In [ ]:
viz.logreg_zero_path(c.X_train, c.y_train, c.feature_names)


## 7. Fashion-MNIST

28×28 greyscale clothes, 10 classes. First run downloads Zalando's files into `data_cache/` (~30MB). Everything below this cell trains on this split.

In [ ]:
DATA = load_fashion_split(n_train=N_TRAIN, n_val=N_VAL, n_test=N_TEST, seed=SEED)
print("train", DATA.X_train.shape, "val", DATA.X_val.shape, "test", DATA.X_test.shape)

n, side = 40, 28
fig, axes = plt.subplots(4, 10, figsize=(11, 4.4), layout="constrained")
for ax, img, lab in zip(axes.ravel(), DATA.X_train[:n], DATA.y_train[:n]):
    ax.imshow(img.reshape(side, side), cmap="gray")
    ax.set_title(FASHION_LABELS[int(lab)], fontsize=7)
    ax.axis("off")
fig.suptitle("Fashion-MNIST train subset")
plt.show()


## 8. Unregularized MLP — overfit baseline

Fat 256-256, small \(n\). Learning curves + a dense first-layer `|W|`. Later cells overlay against this via `MODELS["none"]`.

In [ ]:
model_none, hist_none, met_none = fit_mlp("none", restore_best=False, seed=0)
viz.first_layer_abs(MODELS["none"], title="none: first layer is dense")


## 9. \(L_2\) weight decay

`AdamW` + `weight_decay` on `ndim>1` only (biases skipped).

**Look at:** `|w|` histogram / CDF vs `none`. L2 shrinks the tail, does **not** create a spike at 0.

**Knob:** `WD`.

In [ ]:
WD = 0.04  # try 1e-3, 0.04, 0.2
_ = fit_mlp("l2", weight_decay=WD, seed=1)
if "none" in MODELS:
    print(f"||W||_2 ratio l2/none = {param_l2(MODELS['l2']) / param_l2(MODELS['none']):.3f}")
    viz.weight_sparsity({"none": MODELS["none"], "l2": MODELS["l2"]}, title="L2 shrinks |w|, no extra zeros")
show_board()


## 10. \(L_1\) on weights — null weights in a net

Loss `+= α Σ|W|`. Same CDF as above: L1 should **jump** at small \(t\). The right panel of `first_layer_abs` is a hole-punch mask (`|W|<1e-3` in white).

Adam won't hit machine-zero like coordinate-descent lasso; "null" here is a pile-up near 0. Compare `frac<1e-3` vs `none`.

**Knob:** `L1`. Try `3e-3` if 8 epochs isn't enough to see holes.

In [ ]:
L1 = 8e-4  # try 1e-4, 8e-4, 3e-3
_ = fit_mlp("l1", l1=L1, seed=2)
print("weight L1", float(weight_l1(MODELS["l1"]).detach()))
cmp = {k: MODELS[k] for k in ("none", "l2", "l1") if k in MODELS}
viz.weight_sparsity(cmp, title="L1 vs L2 vs none — L1 CDF jumps")
viz.first_layer_abs(MODELS["l1"], title="L1: white pixels are near-zero weights")
show_board()


## 11. Dropout (ch. 7.12)

Inverted dropout: rescale at train, identity at eval.

**Look at the heatmap:** each row is one train-mode forward of the *same* image. Black columns flicker (dropped units). Bottom row = eval, fully dense — the ensemble average.

**Knob:** `DROPOUT`.

In [ ]:
DROPOUT = 0.5  # try 0.2, 0.5, 0.8
_ = fit_mlp("dropout", dropout=DROPOUT, seed=3)
viz.dropout_passes(MODELS["dropout"], DATA.X_test, title=f"dropout p={DROPOUT}: mask flickers at train, dense at eval")
show_board()


## 12. Input noise (ch. 7.5)

Gaussian noise on \(x\) at train time. Infinitesimal ≈ weight decay (Bishop); finite is strictly stronger.

**Look at:** clean vs noised pixels. That's the regularizer — you train on the bottom row.

**Knob:** `NOISE` (pixels in \([0,1]\)).

In [ ]:
NOISE = 0.15  # try 0.05, 0.15, 0.4
viz.noise_grid(DATA.X_train, NOISE, n=8)
_ = fit_mlp("input_noise", input_noise=NOISE, seed=4)
show_board()


## 13. Label smoothing (ch. 7.5.1)

Replace one-hot \(y\) with \((1-\epsilon)y + \epsilon/K\). Stops softmax logits going to \(\pm\infty\).

**Look at:** histogram of max softmax on test. Smoothed net should have less mass piled on 1.0.

**Knob:** `EPS`.

In [ ]:
EPS = 0.1  # try 0.05, 0.1, 0.3
_ = fit_mlp("label_smooth", label_smoothing=EPS, seed=6)
cmp = {k: MODELS[k] for k in ("none", "label_smooth") if k in MODELS}
viz.confidence_hist(cmp, DATA.X_test[:800])
show_board()


## 14. Early stopping (ch. 7.8)

Track val, restore best snapshot, stop after `patience` non-improvements. Under quadratic \(J\)+GD this is equivalent to \(L_2\) (Bishop / Sjöberg–Ljung).

**Look at:** dotted line = `best_epoch`. Overlay vs `none` if you ran §8. This cell uses extra epochs so stopping can fire.

**Knobs:** `PATIENCE`, `MIN_EPOCH`.

In [ ]:
PATIENCE = 4
MIN_EPOCH = 4
_ = fit_mlp(
    "early_stop",
    epochs=max(EPOCHS, 16),
    early_stop_patience=PATIENCE,
    early_stop_min_epoch=MIN_EPOCH,
    restore_best=True,
    seed=5,
)
print("best_epoch", HISTS["early_stop"].best_epoch, "stopped", HISTS["early_stop"].stopped_epoch)
ov = {k: HISTS[k] for k in ("none", "early_stop") if k in HISTS}
viz.overlay_val(ov, title="early stop restores the red dot, not the last epoch")
show_board()


Stack dropout + L2 if you want.

In [ ]:
_ = fit_mlp("l2+dropout", dropout=0.4, weight_decay=0.02, seed=7)
show_board()


## 15. Parameter sharing — CNN vs MLP (ch. 7.9)

Same 3×3 kernel at every location.

**Look at:** (1) param-count bar, (2) the 16 learned kernels — those *are* the shared weights.

In [ ]:
IMAGES = load_fashion_split(
    n_train=N_TRAIN, n_val=N_VAL, n_test=N_TEST, seed=SEED, as_images=True
)
cnn = TinyCNN(img_size=28)
viz.param_bars({"MLP 256-256": n_params(MLP(784, 10, hidden=HIDDEN)), "TinyCNN": n_params(cnn)})
hist_cnn = train_classifier(
    cnn,
    IMAGES.X_train, IMAGES.y_train, IMAGES.X_val, IMAGES.y_val,
    TrainConfig(epochs=EPOCHS, lr=LR, batch_size=64, optimizer="adamw", seed=11),
)
met_cnn = metrics_for(
    cnn, IMAGES.X_train, IMAGES.y_train, IMAGES.X_val, IMAGES.y_val, IMAGES.X_test, IMAGES.y_test
)
BOARD["cnn"] = {"train": met_cnn.train_acc, "val": met_cnn.val_acc, "test": met_cnn.test_acc, "gap": met_cnn.gap}
MODELS["cnn"] = cnn
HISTS["cnn"] = hist_cnn
print(f"cnn  train={met_cnn.train_acc:.3f}  test={met_cnn.test_acc:.3f}  gap={met_cnn.gap:.3f}")
plot_hist(hist_cnn, "cnn")
viz.conv_kernels(cnn)
show_board()


## 16. Dataset augmentation (ch. 7.4)

Manufacture \((x,y)\) by transforming \(x\) without changing the label. \(\pm\) pixel rolls. Do **not** flip — shirts vs not (and 6/9) silently poisons the label.

**Look at:** top original, bottom rolled. That's the extra train set.

**Knobs:** `MAX_SHIFT`, `N_COPIES`.

In [ ]:
MAX_SHIFT = 2     # try 1, 2, 4
N_COPIES = 2

shifted = shift_images(IMAGES.X_train, max_shift=MAX_SHIFT, seed=20)
viz.shift_grid(IMAGES.X_train, shifted, n=8)

aug_X = [IMAGES.X_train]
aug_y = [IMAGES.y_train]
for i in range(N_COPIES):
    aug_X.append(shift_images(IMAGES.X_train, max_shift=MAX_SHIFT, seed=20 + i))
    aug_y.append(IMAGES.y_train)
X_aug, y_aug = np.concatenate(aug_X), np.concatenate(aug_y)
print("train size", len(IMAGES.y_train), "→", len(y_aug))

cnn_aug = TinyCNN(img_size=28)
hist_aug = train_classifier(
    cnn_aug, X_aug, y_aug, IMAGES.X_val, IMAGES.y_val,
    TrainConfig(epochs=EPOCHS, lr=LR, batch_size=64, optimizer="adamw", seed=12),
)
met_aug = metrics_for(
    cnn_aug, IMAGES.X_train, IMAGES.y_train, IMAGES.X_val, IMAGES.y_val, IMAGES.X_test, IMAGES.y_test
)
BOARD["cnn+aug"] = {"train": met_aug.train_acc, "val": met_aug.val_acc, "test": met_aug.test_acc, "gap": met_aug.gap}
MODELS["cnn+aug"] = cnn_aug
print(f"cnn+aug  train={met_aug.train_acc:.3f}  test={met_aug.test_acc:.3f}  gap={met_aug.gap:.3f}")
plot_hist(hist_aug, "cnn+aug")
show_board()


## 17. Sparse *representations* (ch. 7.10)

| | what is zero | mechanism |
|---|---|---|
| sparse **parameters** | weights | \(L_1\) on \(W\) (§10) |
| sparse **activations** | hidden \(h\) | \(L_1\) on \(h\) (this cell) |

**Look at:** hist piles at 0, and the activation heatmap goes dark (units off).

**Knob:** `ACT_L1`.

In [ ]:
ACT_L1 = 0.15  # try 0, 0.05, 0.15, 0.4

def hidden_of(act_l1: float, name: str):
    torch.manual_seed(30)
    model = MLP(DATA.n_features, DATA.n_classes, hidden=(64, 64), activation="tanh")
    train_classifier(
        model, DATA.X_train, DATA.y_train, DATA.X_val, DATA.y_val,
        TrainConfig(
            epochs=EPOCHS, lr=LR, batch_size=64, optimizer="adamw",
            activation_l1=act_l1, seed=30,
        ),
    )
    model.eval()
    with torch.no_grad():
        _, h = model(torch.from_numpy(DATA.X_test[:200]), return_hidden=True)
    h = h.numpy()
    print(f"{name:18} mean|h|={np.mean(np.abs(h)):.3f}  frac≈0={np.mean(np.abs(h)<1e-2):.3f}")
    return h

h0 = hidden_of(0.0, "tanh")
h1 = hidden_of(ACT_L1, "tanh + L1 on h")
viz.hidden_sparsity(h0, h1, "tanh", f"L1={ACT_L1} on h")


## 18. Bagging (ch. 7.11)

\(k\) models on bootstrap resamples, average softmaxes. Explicit ensemble, unlike dropout.

**Look at:** member test acc vs the bag. Bag should beat the typical member.

**Knob:** `N_BAG`.

In [ ]:
N_BAG = 3  # try 2, 3, 5

members = []
member_test = []
for i in range(N_BAG):
    idx = bootstrap_indices(len(DATA.X_train), seed=20 + i)
    m = MLP(DATA.n_features, DATA.n_classes, hidden=HIDDEN)
    train_classifier(
        m,
        DATA.X_train[idx], DATA.y_train[idx], DATA.X_val, DATA.y_val,
        TrainConfig(epochs=EPOCHS, lr=LR, batch_size=64, optimizer="adamw", seed=20 + i),
    )
    members.append(m)
    te = metrics_for(m, DATA.X_train, DATA.y_train, DATA.X_val, DATA.y_val, DATA.X_test, DATA.y_test).test_acc
    member_test.append(te)
    print(f"  member {i} test={te:.3f}")

bag_tr = float(np.mean(bagged_proba(members, DATA.X_train).argmax(1) == DATA.y_train))
bag_va = float(np.mean(bagged_proba(members, DATA.X_val).argmax(1) == DATA.y_val))
bag_te = float(np.mean(bagged_proba(members, DATA.X_test).argmax(1) == DATA.y_test))
BOARD[f"bag x{N_BAG}"] = {"train": bag_tr, "val": bag_va, "test": bag_te, "gap": bag_tr - bag_te}
print(f"{f'bag x{N_BAG}':18} train={bag_tr:.3f}  val={bag_va:.3f}  test={bag_te:.3f}  gap={bag_tr-bag_te:.3f}")
viz.bag_bars(member_test, bag_te, bag_tr)
show_board()


## 19. Adversarial training / FGSM (ch. 7.13)

\[
x_{\mathrm{adv}} = x + \varepsilon\,\mathrm{sign}(\nabla_x J)
\]

**Look at the triplets:** clean / signed gradient (looks like noise, isn't) / adversarial. Red caption = class flipped. Then bars: train on FGSM, the same attack does less.

**Knob:** `ADV_EPS`.

In [ ]:
ADV_EPS = 0.12  # try 0.03, 0.12, 0.25

def acc(model, X, y, batch=128):
    model.eval()
    n = 0
    correct = 0
    xt = torch.from_numpy(np.ascontiguousarray(X))
    yt = torch.from_numpy(np.ascontiguousarray(y)).long()
    for i in range(0, len(X), batch):
        pred = model(xt[i:i + batch]).argmax(1)
        correct += int((pred == yt[i:i + batch]).sum())
        n += pred.numel()
    return correct / n


def fgsm_acc(model, X, y, eps, batch=128):
    model.eval()
    n = 0
    correct = 0
    xt = torch.from_numpy(np.ascontiguousarray(X))
    yt = torch.from_numpy(np.ascontiguousarray(y)).long()
    for i in range(0, len(X), batch):
        xb, yb = xt[i:i + batch], yt[i:i + batch]
        adv = fgsm(model, xb, yb, eps)
        pred = model(adv).argmax(1)
        correct += int((pred == yb).sum())
        n += yb.numel()
    return correct / n

rows = []
trained = {}
for name, adv in [("clean train", 0.0), ("FGSM train", ADV_EPS)]:
    m = TinyCNN(img_size=28)
    train_classifier(
        m, IMAGES.X_train, IMAGES.y_train, IMAGES.X_val, IMAGES.y_val,
        TrainConfig(
            epochs=EPOCHS, lr=LR, batch_size=64, optimizer="adamw",
            adversarial_eps=adv, seed=40,
        ),
    )
    trained[name] = m
    rows.append({
        "name": name,
        "clean": acc(m, IMAGES.X_test, IMAGES.y_test),
        "fgsm": fgsm_acc(m, IMAGES.X_test, IMAGES.y_test, ADV_EPS),
    })
    print(f"{name:12}  clean={rows[-1]['clean']:.3f}  FGSM={rows[-1]['fgsm']:.3f}")

print("\nattack on the clean net (captions: true → pred, red = flipped):")
viz.fgsm_triplets(trained["clean train"], IMAGES.X_test, IMAGES.y_test, ADV_EPS, FASHION_LABELS, n=6)

fig, ax = plt.subplots(figsize=(6.4, 3.8))
x = np.arange(len(rows))
ax.bar(x - 0.18, [r["clean"] for r in rows], 0.36, label="clean test")
ax.bar(x + 0.18, [r["fgsm"] for r in rows], 0.36, label="FGSM test", color="#d62728")
ax.set_xticks(x)
ax.set_xticklabels([r["name"] for r in rows])
ax.set_ylim(0, 1.05)
ax.legend(frameon=False)
ax.set_title(rf"FGSM $\varepsilon={ADV_EPS}$")
plt.show()


## 20. Scoreboard

`gap = train − test` on clean Fashion. Re-run after any training cell.

In [ ]:
show_board()


## 21. Things to try

- `LASSO_ALPHA` / `C_L1` until the red × appear / disappear.
- `L1=3e-3` on the MLP if the hole-punch mask is still dark at 8 epochs.
- `DROPOUT=0.8` and stare at the mask heatmap.
- `ADV_EPS=0.25` — more visible perturbation, more flips.
- `EPOCHS=25`, `N_TRAIN=1500` for `run.py`-like numbers.

Interview list: `NOTES.md`.